# Практика 7. Обучение без учителя — версия с решениями

> Этот файл предназначен преподавателю: все задания заполнены, добавлены комментарии к разбору. Студентам выдаётся версия без суффикса `_solutions`.

# Практика 7. Обучение без учителя

**Курс «Машинное обучение» · Часть 1, занятие 7**

План:

1. Кластеризация: K-means, иерархическая, DBSCAN — и где каждая ломается
2. Как выбирать число кластеров
3. Снижение размерности: PCA, t-SNE, UMAP
4. Поиск аномалий

> UMAP в Colab ставится так: `!pip install -q umap-learn`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs, make_moons, load_digits, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    silhouette_score, adjusted_rand_score, calinski_harabasz_score
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("готово")

---
## 1. Где ломается K-means

Сгенерируем три набора данных: удобный для K-means и два неудобных.

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=500, centers=4, cluster_std=1.0,
                              random_state=RANDOM_STATE)

X_moons_, y_moons_ = make_moons(n_samples=500, noise=0.06,
                                random_state=RANDOM_STATE)

# Кластеры разной плотности и размера
X_uneven, y_uneven = make_blobs(
    n_samples=[400, 100, 40], centers=[[0, 0], [6, 6], [-5, 6]],
    cluster_std=[1.2, 0.4, 2.5], random_state=RANDOM_STATE,
)

datasets = {
    "шарообразные": (X_blobs, y_blobs, 4),
    "полумесяцы": (X_moons_, y_moons_, 2),
    "разная плотность": (X_uneven, y_uneven, 3),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (name, (Xd, yd, k)) in zip(axes, datasets.items()):
    ax.scatter(Xd[:, 0], Xd[:, 1], c=yd, cmap="tab10", s=18, edgecolors="none")
    ax.set_title(f"{name} (истинных кластеров: {k})")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))

for col, (name, (Xd, yd, k)) in enumerate(datasets.items()):
    Xs = StandardScaler().fit_transform(Xd)

    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Xs)
    axes[0, col].scatter(Xd[:, 0], Xd[:, 1], c=km.labels_, cmap="tab10",
                         s=18, edgecolors="none")
    axes[0, col].set_title(f"K-means: {name}\nARI = {adjusted_rand_score(yd, km.labels_):.2f}")

    db = DBSCAN(eps=0.3, min_samples=8).fit(Xs)
    n_found = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    noise = (db.labels_ == -1).sum()
    axes[1, col].scatter(Xd[:, 0], Xd[:, 1], c=db.labels_, cmap="tab10",
                         s=18, edgecolors="none")
    axes[1, col].set_title(f"DBSCAN: кластеров {n_found}, шума {noise}\n"
                           f"ARI = {adjusted_rand_score(yd, db.labels_):.2f}")

plt.tight_layout()
plt.show()

print("ARI (скорректированный индекс Ранда) сравнивает разбиение с истинным:")
print("1.0 — полное совпадение, 0.0 — случайное разбиение.")

Картина ровно та, что обещала лекция:

- На **шарообразных** кластерах K-means работает идеально
- На **полумесяцах** он проваливается: разрезает их пополам прямыми линиями. DBSCAN справляется
- На кластерах **разной плотности** оба испытывают трудности, но по-разному

### Задание 1

Подберите параметры DBSCAN для третьего набора («разная плотность»):

1. Переберите `eps` и `min_samples` по сетке
2. Для каждой пары посчитайте ARI и число найденных кластеров
3. Найдите лучшую комбинацию и нарисуйте результат
4. Объясните, почему одного набора параметров не хватает для всех трёх кластеров

Сохраните: `best_eps` и `best_min_samples` — лучшие параметры, `best_ari` — качество с ними.

> **Что посмотреть:** `DBSCAN(eps=..., min_samples=...).fit_predict(X)` возвращает метки, где `-1` означает шум — поэтому число кластеров считают как `len(set(labels)) - (1 if -1 in labels else 0)`. Качество разбиения — `adjusted_rand_score(истинные, предсказанные)`. Лучшую строку таблицы даёт `grid.loc[grid["ARI"].idxmax()]`, а тепловую карту по сетке — `df.pivot(index=, columns=, values=)` плюс `sns.heatmap`.

In [ ]:
Xu = StandardScaler().fit_transform(X_uneven)

eps_grid = np.arange(0.15, 0.85, 0.05)
min_samples_grid = [3, 5, 8, 12, 20]

rows = []
for eps in eps_grid:
    for ms in min_samples_grid:

        labels = DBSCAN(eps=eps, min_samples=ms).fit_predict(Xu)
        ari = adjusted_rand_score(y_uneven, labels)

        # Метка -1 у DBSCAN означает шум, кластером она не считается
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        rows.append({"eps": round(eps, 2), "min_samples": ms,
                     "кластеров": n_clusters,
                     "шума": int((labels == -1).sum()),
                     "ARI": round(ari, 3)})

grid = pd.DataFrame(rows)
best = grid.loc[grid["ARI"].idxmax()]
print("лучшая комбинация:")
print(best.to_string())

pivot = grid.pivot(index="min_samples", columns="eps", values="ARI")
plt.figure(figsize=(11, 3.5))
sns.heatmap(pivot, cmap="RdYlGn", vmin=0, vmax=1, cbar_kws={"label": "ARI"})
plt.title("Качество DBSCAN по сетке параметров")
plt.show()

best_eps = best["eps"]
best_min_samples = int(best["min_samples"])
best_ari = best["ARI"]

print()
print("Одного набора параметров на все три кластера не хватает: eps задаёт")
print("единый радиус окрестности, а плотный кластер и разреженный требуют разного.")
print("Малый eps разрежет разреженный на куски или объявит шумом, большой —")
print("склеит плотный с соседями.")
print()
print("Эту проблему и решает HDBSCAN: он строит иерархию плотностей и выбирает")
print("подходящий уровень для каждого кластера отдельно.")


In [ ]:
# Проверка задания 1
assert round(float(best_eps), 2) == 0.40, f"best_eps = {best_eps}"
assert int(best_min_samples) == 8, f"best_min_samples = {best_min_samples}"
assert round(float(best_ari), 2) == 0.97, f"best_ari = {best_ari}"
print("Задание 1 — верно ✓")

> **Для преподавателя.** Если установлен hdbscan (входит в свежие версии scikit-learn как sklearn.cluster.HDBSCAN), полезно показать его результат на этих же данных — контраст обычно наглядный.

---
## 2. Сколько брать кластеров

In [ ]:
Xs_blobs = StandardScaler().fit_transform(X_blobs)

k_range = range(2, 11)
inertia, silhouette, calinski = [], [], []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Xs_blobs)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(Xs_blobs, km.labels_))
    calinski.append(calinski_harabasz_score(Xs_blobs, km.labels_))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(k_range, inertia, marker="o", color="#2B5CE6")
axes[0].set_title("Метод локтя")
axes[0].set_xlabel("число кластеров"); axes[0].set_ylabel("сумма квадратов расстояний")

axes[1].plot(k_range, silhouette, marker="o", color="#1B8A5A")
axes[1].axvline(list(k_range)[int(np.argmax(silhouette))], ls="--", color="#C0393B")
axes[1].set_title("Силуэт (чем больше, тем лучше)")
axes[1].set_xlabel("число кластеров")

axes[2].plot(k_range, calinski, marker="o", color="#B4760B")
axes[2].set_title("Индекс Калинского–Харабаса")
axes[2].set_xlabel("число кластеров")
plt.tight_layout()
plt.show()

print(f"силуэт максимален при k = {list(k_range)[int(np.argmax(silhouette))]}")
print(f"истинное число кластеров: 4")
print()
print("Локоть на первом графике заметен около k=4, но точку перегиба")
print("часто трудно определить однозначно. Силуэт даёт более чёткий ответ.")

**Важная оговорка.** Здесь мы знаем правильный ответ, потому что данные синтетические. На реальных задачах проверить не с чем, и решающим критерием становится интерпретируемость: можно ли описать каждый кластер словами.

---
## 3. Снижение размерности

Возьмём рукописные цифры: 64 признака (изображение 8×8), 10 классов.

In [ ]:
digits = load_digits()
X_dig, y_dig = digits.data, digits.target
print(f"объектов: {X_dig.shape[0]}, признаков: {X_dig.shape[1]}, классов: {len(set(y_dig))}")

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, img, label in zip(axes.ravel(), digits.images, y_dig):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(label), fontsize=9)
    ax.axis("off")
plt.suptitle("Как выглядят данные")
plt.tight_layout()
plt.show()

In [ ]:
X_dig_s = StandardScaler().fit_transform(X_dig)

pca_full = PCA().fit(X_dig_s)
explained = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(explained) + 1), explained, color="#2B5CE6")
for level in (0.8, 0.9, 0.95):
    n_comp = int(np.searchsorted(explained, level) + 1)
    plt.axhline(level, ls=":", color="#A8B0BC")
    plt.axvline(n_comp, ls="--", color="#C0393B", alpha=0.5)
    plt.text(n_comp + 1, level - 0.05, f"{level:.0%} — {n_comp} компонент", fontsize=9)

plt.xlabel("число компонент")
plt.ylabel("доля объяснённой дисперсии")
plt.title("Сколько компонент нужно")
plt.show()

print(f"исходно признаков: {X_dig.shape[1]}")
print(f"для 95% дисперсии достаточно: {int(np.searchsorted(explained, 0.95) + 1)}")

### Задание 2

Проверьте, помогает ли PCA качеству классификации:

1. Обучите логистическую регрессию на всех 64 признаках
2. Обучите её же на 10, 20, 30, 40 главных компонентах
3. Постройте график качества от числа компонент
4. Ответьте: где оптимум и почему сжатие может **улучшать** качество

Сохраните: `base_score` — accuracy на всех 64 признаках, `best_n_components` — число компонент с лучшим качеством, `best_pca_score` — это качество.

> **Что посмотреть:** `PCA(n_components=n)` ставится в `Pipeline` между `StandardScaler` и моделью — иначе сжатие будет считаться по всем данным сразу и это будет утечкой. Логистической регрессии на этих данных нужен запас итераций: `max_iter=3000`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

base_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
])
base_score = cross_val_score(base_pipe, X_dig, y_dig, cv=cv, n_jobs=-1).mean()
print(f"все 64 признака: accuracy = {base_score:.4f}")

n_components = [5, 10, 15, 20, 30, 40, 50, 64]
scores = []
for n in n_components:

    # PCA обязан быть ВНУТРИ пайплайна: иначе компоненты посчитаются
    # по всем фолдам сразу, и это будет утечкой
    pipe = Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=n, random_state=RANDOM_STATE)),
        ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ])
    s = cross_val_score(pipe, X_dig, y_dig, cv=cv, n_jobs=-1).mean()

    scores.append(s)
    print(f"{n:3} компонент:    accuracy = {s:.4f}")

plt.figure(figsize=(10, 4.5))
plt.plot(n_components, scores, marker="o", color="#2B5CE6", label="с PCA")
plt.axhline(base_score, ls="--", color="#C0393B",
            label=f"все 64 признака ({base_score:.3f})")
plt.xlabel("число компонент")
plt.ylabel("accuracy на кросс-валидации")
plt.title("Помогает ли сжатие")
plt.legend()
plt.show()

best_idx = int(np.argmax(scores))
best_n_components = n_components[best_idx]
best_pca_score = scores[best_idx]
print(f"\nлучшее качество при {best_n_components} компонентах: {best_pca_score:.4f}")

print()
print("Сжатие может улучшать качество по трём причинам:")
print("  отбрасываются направления с малой дисперсией — там в основном шум")
print("  признаков становится меньше, и модели труднее переобучиться")
print("  уходит мультиколлинеарность: компоненты ортогональны, а соседние")
print("  пиксели сильно коррелируют между собой")
print()
print("Побочная выгода — скорость. Плата — интерпретируемость: компонента")
print("это линейная комбинация всех пикселей, объяснить её словами трудно.")


In [ ]:
# Проверка задания 2
assert round(float(base_score), 2) == 0.97, f"base_score = {base_score}"
assert int(best_n_components) == 50, f"best_n_components = {best_n_components}"
assert round(float(best_pca_score), 2) == 0.97, f"best_pca_score = {best_pca_score}"
assert best_pca_score >= base_score,     "на этих данных сжатие не должно ухудшать качество"
print("Задание 2 — верно ✓")

> **Для преподавателя.** Если выигрыш окажется небольшим — это нормально для этого датасета. Эффект PCA гораздо ярче при сотнях коррелирующих признаков. Можно предложить студентам повторить на breast_cancer, где 30 признаков сильно связаны между собой.

---
## 4. Поиск аномалий

In [ ]:
# Смешаем нормальные данные с аномалиями
rng = np.random.default_rng(RANDOM_STATE)
X_normal, _ = make_blobs(n_samples=600, centers=[[0, 0], [4, 4]],
                         cluster_std=0.9, random_state=RANDOM_STATE)
X_anomaly = rng.uniform(low=-6, high=10, size=(30, 2))

X_mix = np.vstack([X_normal, X_anomaly])
y_true = np.hstack([np.zeros(len(X_normal)), np.ones(len(X_anomaly))])

iso = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
pred = iso.fit_predict(X_mix)          # -1 — аномалия, 1 — норма
scores_iso = -iso.score_samples(X_mix)  # чем больше, тем аномальнее

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(X_mix[:, 0], X_mix[:, 1], c=y_true, cmap="coolwarm", s=18)
axes[0].set_title("истина (красное — аномалии)")

axes[1].scatter(X_mix[:, 0], X_mix[:, 1], c=(pred == -1), cmap="coolwarm", s=18)
axes[1].set_title("предсказание Isolation Forest")

sc = axes[2].scatter(X_mix[:, 0], X_mix[:, 1], c=scores_iso, cmap="viridis", s=18)
axes[2].set_title("степень аномальности")
plt.colorbar(sc, ax=axes[2])
plt.tight_layout()
plt.show()

from sklearn.metrics import roc_auc_score, precision_score, recall_score
detected = (pred == -1).astype(int)
print(f"ROC-AUC по непрерывной оценке: {roc_auc_score(y_true, scores_iso):.4f}")
print(f"precision: {precision_score(y_true, detected):.3f}")
print(f"recall:    {recall_score(y_true, detected):.3f}")

### Задание 3

Примените поиск аномалий к реальным данным:

1. Возьмите `load_breast_cancer` и обучите `IsolationForest` **только на здоровых** наблюдениях
2. Проверьте, насколько хорошо метод отличает больных как «аномалии»
3. Сравните с `LocalOutlierFactor`
4. Сделайте вывод: применим ли поиск аномалий к этой задаче и когда он оправдан

Сохраните: `auc_iforest` и `auc_lof` — качество двух методов, `sup_auc` — качество обычного классификатора с разметкой.

> **Что посмотреть:** `IsolationForest(contamination=0.05)` обучается **только на норме**: `.fit(X_норма)`. `LocalOutlierFactor` для этого требует `novelty=True`. Оба дают оценку через `.score_samples(X)`, и чем она **меньше**, тем объект аномальнее — поэтому перед `roc_auc_score` знак меняют на противоположный.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
Xc, yc = cancer.data, cancer.target      # 1 — доброкачественная, 0 — злокачественная

# «Нормой» считаем доброкачественные наблюдения
X_benign, X_malign = Xc[yc == 1], Xc[yc == 0]
X_fit, X_benign_test = train_test_split(X_benign, test_size=0.3,
                                        random_state=RANDOM_STATE)
X_eval = np.vstack([X_benign_test, X_malign])
y_eval = np.hstack([np.zeros(len(X_benign_test)), np.ones(len(X_malign))])

scaler = StandardScaler().fit(X_fit)
X_fit_s, X_eval_s = scaler.transform(X_fit), scaler.transform(X_eval)
print(f"учимся на {len(X_fit)} здоровых, проверяем на {len(X_eval)} смешанных")

iso = IsolationForest(contamination=0.05,
                      random_state=RANDOM_STATE).fit(X_fit_s)
iso_scores = -iso.score_samples(X_eval_s)

# novelty=True нужен, чтобы метод умел оценивать НОВЫЕ объекты
lof = LocalOutlierFactor(n_neighbors=20, novelty=True).fit(X_fit_s)
lof_scores = -lof.score_samples(X_eval_s)

auc_iforest = roc_auc_score(y_eval, iso_scores)
auc_lof = roc_auc_score(y_eval, lof_scores)

print(f"\n{'метод':24} {'ROC-AUC':>9}")
print(f"{'Isolation Forest':24} {auc_iforest:9.4f}")
print(f"{'LocalOutlierFactor':24} {auc_lof:9.4f}")

# Для сравнения — обычный классификатор, который видел оба класса
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    Xc, yc, test_size=0.3, random_state=RANDOM_STATE, stratify=yc)
clf = Pipeline([("scale", StandardScaler()),
                ("m", LogisticRegression(max_iter=3000))]).fit(Xc_tr, yc_tr)
sup_auc = roc_auc_score(1 - yc_te, clf.predict_proba(Xc_te)[:, 0])
print(f"{'модель с разметкой':24} {sup_auc:9.4f}")

print()
print("Поиск аномалий работает прилично, но модель с разметкой заметно лучше.")
print("Это главный вывод: если примеры редкого класса есть — это задача")
print("классификации с дисбалансом, и решать её надо весами классов,")
print("правильной метрикой и подбором порога.")
print()
print("Поиск аномалий оправдан, когда размеченных примеров нет вовсе,")
print("аномалии разнородны, и ловить нужно то, чего раньше не случалось.")


In [ ]:
# Проверка задания 3
assert round(float(auc_iforest), 2) == 0.96, f"auc_iforest = {auc_iforest}"
assert round(float(auc_lof), 2) == 0.95, f"auc_lof = {auc_lof}"
assert round(float(sup_auc), 2) == 1.0, f"sup_auc = {sup_auc}"
assert sup_auc > auc_iforest,     "модель с разметкой обязана обойти поиск аномалий — это главный вывод задания"
print("Задание 3 — верно ✓")

---
## Итоги занятия

- K-means предполагает шарообразные кластеры сопоставимого размера — за пределами этого он ломается
- DBSCAN находит кластеры любой формы и помечает шум, но чувствителен к параметрам и не любит разную плотность
- Число кластеров подсказывают силуэт и локоть, но решает **интерпретируемость**
- PCA сжимает линейно и может улучшать качество, убирая шум
- t-SNE и UMAP — только для картинок; **расстояния на них не значат ничего**
- Поиск аномалий нужен, когда разметки редкого класса нет. Если она есть — это классификация с дисбалансом

## Что дальше

Практика 8 — собираем всё в один проект: от сырых данных до работающего сервиса.